# 01 - Synthetic data for the Fanimation Support Command Center
Generates 15 days (2026-10-13 to 2026-10-27, UTC, 5-minute grain) of interconnected data into `classic_stable_l0m1dn_catalog.support_command_center`:
`regions`, `release_schedule`, `concurrent_streams`, `service_metrics`, `payment_transactions`, `support_tickets`, `social_posts`, `change_events`, `incidents`.

Scripted demo day **2026-10-27**:
* 09:00 APAC login failures (AUTH-503) - monitoring catches it fast
* 14:00 *Blade of the Crimson Moon* finale simulcast - benign traffic spike
* 15:30 Android TV player SDK v2.3.0 deploy, 15:36 BR/MX/AR Android TV Error 1002 - quiet session drops, **customer-detected 40+ min before monitoring** (declared 16:25)
* 18:00 *Spirit Ronin* E3 Spanish subtitles missing - support escalation only (declared 19:40)
* 20:00 Adyen EU endpoint migration, DE/AT payment failures (declared 20:50)

Historical incidents: Oct 16 NA auth, Oct 20 EMEA CDN, Oct 23 FR subtitle sync, Oct 25 PH/ID Google Play billing.

In [0]:
import numpy as np, pandas as pd, random
CAT="classic_stable_l0m1dn_catalog"; SCH="support_command_center"; FQ=f"{CAT}.{SCH}"
rng=np.random.default_rng(42); random.seed(42)
start=pd.Timestamp("2026-10-13 00:00:00"); T=15*288
TS=pd.date_range(start, periods=T, freq="5min")
def iv(t): return int((pd.Timestamp(t)-start)/pd.Timedelta(minutes=5))
def window(t0,t1,rin=2,rout=3):
    a=np.zeros(T); i0,i1=iv(t0),iv(t1)
    for i in range(i0,min(T,i1+rout)):
        a[i]= (i-i0+1)/rin if i<i0+rin else (1.0 if i<i1 else max(0,1-(i-i1+1)/rout))
    return a

countries=[("US","United States","NA",39.8,-98.6,-5,5200,"English"),("CA","Canada","NA",56.1,-106.3,-5,700,"English"),
("MX","Mexico","LATAM",23.6,-102.5,-6,1300,"Spanish"),("BR","Brazil","LATAM",-14.2,-51.9,-3,1800,"Portuguese"),
("AR","Argentina","LATAM",-38.4,-63.6,-3,450,"Spanish"),("CO","Colombia","LATAM",4.6,-74.1,-5,380,"Spanish"),
("CL","Chile","LATAM",-35.7,-71.5,-3,300,"Spanish"),("GB","United Kingdom","EMEA",55.4,-3.4,0,900,"English"),
("FR","France","EMEA",46.2,2.2,1,1000,"French"),("DE","Germany","EMEA",51.2,10.4,1,850,"German"),
("ES","Spain","EMEA",40.5,-3.7,1,600,"Spanish"),("IT","Italy","EMEA",41.9,12.6,1,450,"Italian"),
("AT","Austria","EMEA",47.5,14.6,1,120,"German"),("SA","Saudi Arabia","EMEA",23.9,45.1,3,250,"Arabic"),
("IN","India","APAC",20.6,78.96,5.5,900,"English"),("JP","Japan","APAC",36.2,138.3,9,400,"Japanese"),
("AU","Australia","APAC",-25.3,133.8,10,350,"English"),("PH","Philippines","APAC",12.9,121.8,8,500,"English"),
("ID","Indonesia","APAC",-0.8,113.9,7,300,"Indonesian"),("KR","South Korea","APAC",35.9,127.8,9,150,"Korean")]
cdf=pd.DataFrame(countries,columns=["country_code","country_name","region_group","lat","lon","utc_offset","subscribers_k","primary_language"])
cdf["subscribers"]=cdf.subscribers_k*1000
C=len(cdf); cidx={c:i for i,c in enumerate(cdf.country_code)}
devices=["Android TV","Fire TV","Roku","Apple TV","iOS","Android","Web","PlayStation","Xbox"]
dshare=np.array([.14,.08,.10,.04,.16,.22,.14,.08,.04]); D=len(devices); didx={d:i for i,d in enumerate(devices)}
share=np.tile(dshare,(C,1))
for c in ["BR","MX","AR"]: share[cidx[c],0]=0.22
share=share/share.sum(1,keepdims=True)

shows=[("Blade of the Crimson Moon",1,"14:00",0.30,10),("Spirit Ronin",1,"18:00",0.12,1),("Iron Samurai Gaiden",2,"14:30",0.12,4),
("Moonlit Idol Stage",3,"16:30",0.08,8),("Mecha Academy Zero",4,"15:00",0.15,6),("Isekai Cafe Chronicles",5,"13:30",0.10,3),
("Shadow Guild Hunters",6,"15:30",0.20,9),("Ocean Spirit Diaries",0,"17:00",0.07,2)]
rel=[]
for d in range(15):
    day=start+pd.Timedelta(days=d); wd=day.weekday()
    for name,wkd_,hm,up,ep0 in shows:
        if wd==wkd_:
            ep=ep0+d//7; u=0.45 if (name.startswith("Blade") and str(day.date())=="2026-10-27") else up
            rel.append((f"REL-{len(rel)+1:03d}",name,int(ep),pd.Timestamp(f"{day.date()} {hm}"),u,"Season Finale" if (name.startswith("Blade") and ep==12) else "New Episode"))
rel_df=pd.DataFrame(rel,columns=["release_id","show_title","episode_number","release_ts","expected_uplift","release_type"])
mins=np.arange(T)*5; uplift=np.zeros(T)
for r in rel_df.itertuples():
    m=mins-iv(r.release_ts)*5
    uplift+=r.expected_uplift*np.where((m>=0)&(m<=360), np.minimum(1,(m+5)/30)*np.exp(-np.maximum(0,m-30)/80),0)

# concurrent streams (time x country x device): diurnal local prime time, weekend lift, trend, simulcast uplift, incidents
mod=(np.arange(T)*5)%1440; day_idx=(np.arange(T)*5)//1440; off=cdf.utc_offset.values
lh=(mod[:,None]/60+off[None,:]); lday=np.floor(lh/24)+day_idx[:,None]; lh=lh%24; wkd=(1+lday)%7
def cd(h,c): x=np.abs(h-c)%24; return np.minimum(x,24-x)
diurnal=0.08+0.92*np.exp(-cd(lh,21)**2/(2*2.8**2))+0.15*np.exp(-cd(lh,13)**2/(2*2.2**2))
base=(cdf.subscribers.values[None,:]*0.075*diurnal*np.where(wkd>=5,1.15,1.0)*(1+0.002*day_idx[:,None])*(1+uplift[:,None]))
noise=rng.lognormal(0,0.03,(T,C,D)); factor=np.ones((T,C,D))
def hit(cs,devs,t0,t1,depth,rin=2,rout=3):
    w_=window(t0,t1,rin,rout)
    for c in cs:
        for d in (devs or devices): factor[:,cidx[c],didx[d]]*=(1-depth*w_)
hit(["JP","IN","AU","PH","ID","KR"],None,"2026-10-27 09:00","2026-10-27 09:45",0.15)
hit(["BR"],["Android TV"],"2026-10-27 15:36","2026-10-27 17:20",0.40,rin=3)
hit(["BR"],["Android TV"],"2026-10-27 16:15","2026-10-27 17:20",0.30)
hit(["MX","AR"],["Android TV"],"2026-10-27 15:36","2026-10-27 17:20",0.18,rin=3)
hit(["MX","ES","AR","CO","CL"],None,"2026-10-27 18:00","2026-10-27 20:05",0.03)
hit(["US","CA"],None,"2026-10-16 02:00","2026-10-16 03:30",0.22)
hit(["GB","FR"],None,"2026-10-20 19:00","2026-10-20 20:15",0.30)
streams_nf=base[:,:,None]*share[None,:,:]*noise
streams=np.round(streams_nf*factor).astype(np.int64); Ls=np.maximum(0,streams_nf*(1-factor))

# internal service telemetry (time x country x domain); note the Error 1002 incident is nearly invisible (1.4x) until 16:15
domains=["Authentication","Billing & Payments","Playback & CDN","Content Ingestion","Subtitles & Localization","Device Apps","Search & Discovery"]
base_err=np.array([0.004,0.010,0.006,0.002,0.001,0.003,0.002]); base_lat=np.array([180,320,140,900,60,210,120])
G=len(domains); gi={g:i for i,g in enumerate(domains)}; err_mult=np.ones((T,C,G))
def emult(cs,dom,t0,t1,mult,rin=2,rout=3):
    w_=window(t0,t1,rin,rout)
    for c in cs: err_mult[:,cidx[c],gi[dom]]*=(1+(mult-1)*w_)
emult(["JP","IN","AU","PH","ID","KR"],"Authentication","2026-10-27 09:00","2026-10-27 09:45",30,rin=1)
emult(["BR"],"Playback & CDN","2026-10-27 15:36","2026-10-27 16:15",1.4,rout=1)
emult(["BR"],"Playback & CDN","2026-10-27 16:15","2026-10-27 17:20",7)
emult(["MX","AR"],"Playback & CDN","2026-10-27 16:15","2026-10-27 17:20",2.2)
emult(["MX","ES","AR","CO","CL"],"Subtitles & Localization","2026-10-27 18:00","2026-10-27 20:05",1.8)
emult(["DE","AT"],"Billing & Payments","2026-10-27 20:00","2026-10-27 20:35",2.0,rout=1)
emult(["DE","AT"],"Billing & Payments","2026-10-27 20:35","2026-10-27 22:10",8)
emult(["US","CA"],"Authentication","2026-10-16 02:20","2026-10-16 03:30",20)
emult(["GB","FR"],"Playback & CDN","2026-10-20 19:00","2026-10-20 20:15",9,rin=1)
emult(["FR"],"Subtitles & Localization","2026-10-23 15:15","2026-10-23 17:45",1.5)
emult(["PH","ID"],"Billing & Payments","2026-10-25 11:00","2026-10-25 12:30",10)
err=np.clip(base_err[None,None,:]*err_mult*rng.lognormal(0,0.12,(T,C,G)),0,0.95)
lat=base_lat[None,None,:]*(1+0.5*(err_mult-1)/np.maximum(err_mult,1))*rng.lognormal(0,0.06,(T,C,G))
req=(streams.sum(2)[:,:,None]*np.array([0.08,0.01,1.0,0.002,0.3,0.2,0.15])[None,None,:]).astype(np.int64)+10
ii,cc,gg=np.meshgrid(np.arange(T),np.arange(C),np.arange(G),indexing="ij")
sm=pd.DataFrame({"ts":TS.values[ii.ravel()],"country_code":cdf.country_code.values[cc.ravel()],"domain":np.array(domains)[gg.ravel()],
 "request_count":req.ravel(),"error_rate":np.round(err.ravel(),5),"p95_latency_ms":np.round(lat.ravel(),1)})
ii,cc,dd=np.meshgrid(np.arange(T),np.arange(C),np.arange(D),indexing="ij")
cs_df=pd.DataFrame({"ts":TS.values[ii.ravel()],"country_code":cdf.country_code.values[cc.ravel()],"device":np.array(devices)[dd.ravel()],
 "concurrent_streams":streams.ravel(),"playback_start_failures":np.round(streams.ravel()*np.repeat(err[:,:,2],D,axis=1).reshape(-1)*0.2+Ls.ravel()*0.05).astype(np.int64)})

In [0]:
rng=np.random.default_rng(7)
gw_fail_mult={}
def pwin(cs,gw,t0,t1,fail,rin=2,rout=2):
    w_=window(t0,t1,rin,rout)
    for c in cs: gw_fail_mult[(c,gw)]=np.maximum(gw_fail_mult.get((c,gw),np.zeros(T)),fail*w_)
pwin(["DE","AT"],"Adyen","2026-10-27 20:00","2026-10-27 20:35",0.06,rout=1)
pwin(["DE","AT"],"Adyen","2026-10-27 20:35","2026-10-27 22:10",0.27)
pwin(["PH","ID"],"Google Play","2026-10-25 11:00","2026-10-25 12:30",0.30)
price={"Stripe":9.99,"Adyen":10.49,"PayPal":9.99,"Apple IAP":11.99,"Google Play":10.99}
eu_adyen={"DE","AT","FR","IT","ES","GB"}
renew=3*(cdf.subscribers.values*0.8/30/288)[None,:]*(0.3+1.4*np.exp(-cd(lh,14)**2/(2*4**2)))
pay_rows=[]; pay_fail=np.zeros((T,C))
for c in cdf.country_code:
    ci=cidx[c]
    gws={"Stripe":.35,"PayPal":.2,"Apple IAP":.2,"Google Play":.25} if c not in eu_adyen else {"Stripe":.15,"Adyen":.3,"PayPal":.2,"Apple IAP":.15,"Google Play":.2}
    for gw,sh in gws.items():
        att=rng.poisson(renew[:,ci]*sh)
        gf=rng.binomial(att,np.clip(0.005+gw_fail_mult.get((c,gw),np.zeros(T)),0,1)); cf=rng.binomial(att-gf,0.025 if gw in ("Stripe","Adyen") else 0.012)
        pay_fail[:,ci]+=gf
        pay_rows.append(pd.DataFrame({"ts":TS,"country_code":c,"gateway":gw,"attempts":att,"successes":att-gf-cf,"customer_declines":cf,"gateway_errors":gf,"avg_price_usd":price[gw]}))
pay_df=pd.concat(pay_rows,ignore_index=True)

# Ticket volume = streams x category contact rate (Poisson) + incident-specific surges
cats={"playback_buffering":("Playback & CDN",0.10,"Playback"),"playback_error_code":("Playback & CDN",0.04,"Playback"),
"login_issue":("Authentication",0.08,"Account & Login"),"password_reset":("Authentication",0.05,"Account & Login"),
"payment_failed":("Billing & Payments",0.04,"Billing"),"billing_question":("Billing & Payments",0.05,"Billing"),
"cancellation_request":("Billing & Payments",0.03,"Billing"),"subtitles_missing":("Subtitles & Localization",0.015,"Subtitles & Audio"),
"subtitles_sync":("Subtitles & Localization",0.015,"Subtitles & Audio"),"dub_quality":("Subtitles & Localization",0.01,"Subtitles & Audio"),
"episode_unavailable":("Content Ingestion",0.02,"Content"),"app_crash":("Device Apps",0.05,"App Issue"),"search_issue":("Search & Discovery",0.02,"App Issue")}
catn=list(cats); rates=np.array([cats[k][1] for k in catn])
cnt=np.random.default_rng(42).poisson((streams/1e4)[:,:,:,None]*rates[None,None,None,:])
t_i,c_i,d_i,k_i=np.nonzero(cnt); reps=cnt[t_i,c_i,d_i,k_i]
base_t=pd.DataFrame({"t":np.repeat(t_i,reps),"c":np.repeat(c_i,reps),"d":np.repeat(d_i,reps),"cat":np.array(catn)[np.repeat(k_i,reps)],"tag":"base"})
inc=[]
def add(tag,cat,cs,devs,t0,t1,lam_fn,rin=2,rout=3):
    w_=window(t0,t1,rin,rout)
    for c in cs:
        for d in (devs or devices):
            n=rng.poisson(lam_fn(cidx[c],didx[d])*w_)
            inc.append(pd.DataFrame({"t":np.repeat(np.arange(T),n),"c":cidx[c],"d":didx[d],"cat":cat,"tag":tag}))
add("auth_apac","login_issue",["JP","IN","AU","PH","ID","KR"],None,"2026-10-27 09:00","2026-10-27 09:45",lambda c,d: streams[:,c,d]*0.0012,rin=1)
add("err1002","playback_error_code",["BR"],["Android TV"],"2026-10-27 15:38","2026-10-27 17:22",lambda c,d: Ls[:,c,d]*0.010+4)
add("err1002","playback_error_code",["MX","AR"],["Android TV"],"2026-10-27 15:40","2026-10-27 17:22",lambda c,d: Ls[:,c,d]*0.010+0.8)
add("subs_ep3","subtitles_missing",["MX","ES","AR","CO","CL"],None,"2026-10-27 18:03","2026-10-27 20:10",lambda c,d: streams[:,c,d]*0.0007,rin=3,rout=4)
add("adyen","payment_failed",["DE","AT"],None,"2026-10-27 20:02","2026-10-27 22:15",lambda c,d: pay_fail[:,c]*0.06+streams[:,c,d]*0.00015,rin=1)
add("auth_na","login_issue",["US","CA"],None,"2026-10-16 02:00","2026-10-16 03:30",lambda c,d: streams[:,c,d]*0.0003)
add("cdn_eu","playback_buffering",["GB","FR"],None,"2026-10-20 19:02","2026-10-20 20:15",lambda c,d: streams[:,c,d]*0.0006)
add("subs_fr","subtitles_sync",["FR"],None,"2026-10-23 15:15","2026-10-23 17:45",lambda c,d: streams[:,c,d]*0.0005)
add("gplay","payment_failed",["PH","ID"],["Android","Android TV"],"2026-10-25 11:00","2026-10-25 12:30",lambda c,d: pay_fail[:,c]*0.08+streams[:,c,d]*0.0002)
tk=pd.concat([base_t]+inc,ignore_index=True)
print(len(tk), tk.tag.value_counts().to_dict())

In [0]:
show_names=[s[0] for s in shows]
base_tpl={
"playback_buffering":[("Video keeps buffering","{show} keeps buffering every few minutes on my {device}. My internet is fine for other apps."),("Stream quality drops","Picture goes blurry and buffers constantly while watching {show}. Using {device}.")],
"playback_error_code":[("Error {code} when playing","I get Error {code} when trying to start {show} on {device}."),("Playback error {code}","Episode won't load, shows error code {code}. Device: {device}.")],
"login_issue":[("Can't sign in","I can't log in to my account on {device}, it says something went wrong."),("Login not working","Keeps asking me to log in again and then fails on {device}.")],
"password_reset":[("Password reset email not received","I requested a password reset but never got the email."),("Reset password","Need help resetting my password, the link expired.")],
"payment_failed":[("Payment failed","My renewal payment failed but my card is fine."),("Card declined","Card declined when upgrading to Mega Fan.")],
"billing_question":[("Charged twice","I think I was charged twice this month for my membership."),("Question about my bill","Why did my price change on the last invoice?")],
"cancellation_request":[("Cancel my membership","Please cancel my subscription before the next billing date."),("How do I cancel","I want to cancel, can't find the option on {device}.")],
"subtitles_missing":[("Subtitles not available","Subtitles are not showing for {show} episode {ep}."),("No subtitle option","The subtitle menu is empty on {show}.")],
"subtitles_sync":[("Subtitles out of sync","Subtitles are a few seconds late on {show} episode {ep}."),("Subs timing wrong","Subtitle timing is off on {show}.")],
"dub_quality":[("Dub audio issue","The English dub audio cuts out in {show} episode {ep}."),("Audio track wrong","Selected dub but it plays Japanese audio on {show}.")],
"episode_unavailable":[("Episode not showing","Episode {ep} of {show} is not available in my region yet?"),("New episode missing","I can't find the new episode of {show}.")],
"app_crash":[("App crashes","The app crashes when I open my watchlist on {device}."),("App freezing","App freezes on the home screen on {device} after the update.")],
"search_issue":[("Search not working","Search doesn't find {show} even though it's on the site."),("Can't find show","Searching for {show} returns no results.")]}
inc_tpl={
"err1002":{"Portuguese":[("Erro 1002 na Android TV","Toda vez que tento assistir {show} aparece Erro 1002 na minha Android TV. Já reiniciei o app e a TV."),("Vídeo para com erro 1002","O episódio começa e depois de alguns segundos cai com erro 1002. Android TV. Terceira vez hoje!")],
  "Spanish":[("Error 1002 en Android TV","No puedo ver {show}, me sale error 1002 en mi Android TV.")],
  "_":[("Error 1002 on Android TV","I keep getting Error 1002 when I try to play {show} on my Android TV. Restarted everything, still broken."),("Playback drops - error code 1002","Video starts then disconnects with error code 1002. Android TV. This is ridiculous, I pay for Mega Fan.")]},
"subs_ep3":{"Spanish":[("Faltan subtítulos en español","El episodio 3 de Spirit Ronin no tiene subtítulos en español. ¿Cuándo lo arreglan?"),("Sin subtítulos episodio 3","Spirit Ronin episodio 3 solo tiene subtítulos en inglés, falta el español latino.")],
  "_":[("Spanish subtitles missing - Spirit Ronin episode 3","Episode 3 of Spirit Ronin has no Spanish subtitles, only English in the menu."),("No subs on Spirit Ronin ep 3","Spirit Ronin Episode 3 - Latin American Spanish subtitles are gone.")]},
"adyen":{"German":[("Zahlung fehlgeschlagen","Meine Kreditkartenzahlung für die Verlängerung wird ständig abgelehnt (payment could not be processed), obwohl die Karte funktioniert."),("Abo-Verlängerung schlägt fehl","Payment failed beim Verlängern meiner Mega Fan Mitgliedschaft. Bank sagt alles ok.")],
  "_":[("Payment declined renewing subscription","My card was declined when renewing Mega Fan, but my bank shows no block. Message: payment could not be processed."),("Payment failed at checkout","Payment failed when upgrading to Ultimate Fan, tried two cards.")]},
"auth_apac":{"_":[("Can't log in - AUTH-503","Login spinner then 'Something went wrong (AUTH-503)' on {device}."),("Login error AUTH-503","Signed out suddenly and now can't log back in, error AUTH-503.")]},
"auth_na":{"_":[("Logged out, can't sign back in","Got logged out and now sign in fails with AUTH-401 on {device}."),("Login failing AUTH-401","Every login attempt says invalid session AUTH-401.")]},
"cdn_eu":{"French":[("Mise en mémoire tampon constante","La vidéo charge sans arrêt, erreur 2001 sur {device}.")],"_":[("Constant buffering - error 2001","Everything buffers and then error 2001 appears on {device}.")]},
"subs_fr":{"French":[("Sous-titres décalés épisode 7","Les sous-titres de Mecha Academy Zero épisode 7 sont décalés de plusieurs secondes.")],"_":[("Subtitles out of sync - Mecha Academy Zero ep 7","Subtitles on Mecha Academy Zero episode 7 are several seconds behind.")]},
"gplay":{"_":[("Google Play payment failed","Google Play says transaction failed, error BILL-207, when renewing Fanimation."),("Can't renew via Google Play","Renewal through Google Play fails with BILL-207.")]}}
lang_of=dict(zip(cdf.country_code,cdf.primary_language))
codes=np.array(["2001","3004","4011","5003","1002"]); cw=np.array([.3,.25,.2,.2,.05])
tk=tk.sample(frac=1,random_state=1).reset_index(drop=True); n=len(tk)
rshow=rng.choice(show_names,n); rep=rng.integers(1,12,n); rcode=rng.choice(codes,n,p=cw); rr=rng.random(n)
cc_arr=cdf.country_code.values[tk.c.values]; dev_arr=np.array(devices)[tk.d.values]
subj=[];body=[]
for i,(cat,tag,cc_,dv) in enumerate(zip(tk.cat.values,tk.tag.values,cc_arr,dev_arr)):
    if tag=="base":
        s,b=base_tpl[cat][int(rr[i]*len(base_tpl[cat]))%len(base_tpl[cat])]; sh=rshow[i]
    else:
        opts=inc_tpl[tag]; lg=lang_of[cc_]
        pool=opts.get(lg) if (lg in opts and rr[i]<0.55) else opts["_"]
        s,b=pool[int(rr[i]*100)%len(pool)]; sh={"subs_ep3":"Spirit Ronin","subs_fr":"Mecha Academy Zero"}.get(tag,rshow[i])
    f=dict(show=sh,ep=int(rep[i]),device=dv,code=rcode[i]); subj.append(s.format(**f)); body.append(b.format(**f))
tk["subject"]=subj; tk["description"]=body
tk["created_at"]=TS.values[tk.t.values]+pd.to_timedelta(rng.integers(0,300,n),unit="s")
tk=tk.sort_values("created_at").reset_index(drop=True); n=len(tk)
tk["ticket_id"]=["CR-"+str(4800000+i) for i in range(n)]
tk["country_code"]=cdf.country_code.values[tk.c.values]; tk["device"]=np.array(devices)[tk.d.values]
tk["language"]=[lang_of[c] for c in tk.country_code]
tk["channel"]=rng.choice(["Chat","Email","In-App","Social"],n,p=[.45,.25,.2,.1])
tk["subscriber_tier"]=rng.choice(["Free","Fan","Mega Fan","Ultimate Fan"],n,p=[.3,.35,.28,.07])
form_map={k:v[2] for k,v in cats.items()}; r2=rng.random(n)
forms=np.array(["Playback","Account & Login","Billing","Subtitles & Audio","Content","App Issue","Other"])
tk["form_category"]=np.where(r2<0.70,[form_map[c] for c in tk.cat],np.where(r2<0.95,"Other",rng.choice(forms,n)))
load=tk.groupby("t").size().reindex(range(T),fill_value=0).rolling(6,min_periods=1).mean()
loadf=np.clip((load/load.median()).values,1,None)**1.2
med={"Chat":3,"Email":90,"In-App":30,"Social":45}
frt=np.array([med[c] for c in tk.channel])*rng.lognormal(0,0.5,n)*loadf[tk.t.values]
tk["first_response_minutes"]=np.round(frt,1)
tk["sla_target_minutes"]=[{"Free":240,"Fan":120,"Mega Fan":60,"Ultimate Fan":30}[x] for x in tk.subscriber_tier]
inc_flag=(tk.tag!="base").values
tk["resolved_at"]=tk.created_at+pd.to_timedelta(frt+rng.lognormal(np.where(inc_flag,0.8,0.3),0.8,n)*60,unit="m")
tk["csat_score"]=np.where(rng.random(n)<0.35,np.clip(np.round(rng.normal(4.3-0.9*inc_flag-0.3*np.log1p(frt/60),0.8)),1,5),np.nan)
tk["reopened"]=rng.random(n)<np.where(inc_flag,0.12,0.04)
tickets=tk[["ticket_id","created_at","country_code","language","device","channel","subscriber_tier","form_category","subject","description","first_response_minutes","sla_target_minutes","resolved_at","csat_score","reopened"]]

In [0]:
W_=T//3; TS15=TS[::3]
gl_diurnal=streams.sum((1,2))[::3]/streams.sum((1,2)).mean(); up15=uplift[::3]; posts=[]
hype=["{show} episode {ep} was INSANE, the animation this week 🔥","Can't believe that ending of {show}. Fanimation simulcast came out right on time!","{show} is the best show of the season, fight me","Watching {show} with the dub tonight, so hyped"]
gen_neg=["Fanimation app on my TV is so slow lately","Why is the Fanimation catalog different in my country","Fanimation raised prices again?","Ads on the free tier are getting out of hand"]
neutral=["Is {show} getting a dub?","Any recs similar to {show}?","When does {show} episode {ep} drop in my timezone?"]
def P(ts,plat,ctry,txt): posts.append((ts,plat,ctry,txt))
for w_ in range(W_):
    t0=TS15[w_]
    for _ in range(rng.poisson(1.6*gl_diurnal[w_]+up15[w_]*35)): P(t0,rng.choice(["Reddit","X"]),rng.choice(cdf.country_code),random.choice(hype).format(show=random.choice(show_names),ep=random.randint(1,12)))
    for _ in range(rng.poisson(0.6*gl_diurnal[w_])): P(t0,rng.choice(["Reddit","X"]),rng.choice(cdf.country_code),random.choice(gen_neg))
    for _ in range(rng.poisson(0.5*gl_diurnal[w_])): P(t0,rng.choice(["Reddit","X"]),rng.choice(cdf.country_code),random.choice(neutral).format(show=random.choice(show_names),ep=random.randint(1,12)))
soc_inc=[("2026-10-27 15:50","2026-10-27 17:35",7,["BR","MX","AR"],["Fanimation down on Android TV? Error 1002 every episode #FanimationDown","Erro 1002 na Fanimation de novo, Android TV não funciona 😡","r/Fanimation megathread: Error 1002 on Android TV in Brazil?","Paying for Mega Fan and can't watch anything, error 1002 on my TV","Anyone else getting error 1002? Phone works, Android TV doesn't"]),
("2026-10-27 09:05","2026-10-27 09:50",4,["JP","IN","AU","PH"],["Fanimation logged me out and AUTH-503 won't let me back in","Is Fanimation login broken in Asia right now?"]),
("2026-10-27 18:15","2026-10-27 20:30",4,["MX","ES","AR","CO"],["Spirit Ronin ep 3 sin subtítulos en español?? @Fanimation","Fanimation olvidó los subtítulos en español de Spirit Ronin episodio 3","No Spanish subs for Spirit Ronin episode 3, really?"]),
("2026-10-27 20:10","2026-10-27 22:15",3,["DE","AT"],["Fanimation Zahlung wird abgelehnt, jemand anderes auch?","Fanimation keeps declining my card on renewal, bank says fine","Payment failed on Fanimation 3 times tonight, what's going on"]),
("2026-10-16 02:05","2026-10-16 03:35",5,["US","CA"],["Fanimation logged everyone out?? AUTH-401","Can't sign into Fanimation, anyone else?"]),
("2026-10-20 19:05","2026-10-20 20:15",4,["GB","FR"],["Fanimation buffering nonstop in the UK tonight","Fanimation en panne? erreur 2001"]),
("2026-10-23 15:30","2026-10-23 17:45",2,["FR"],["Sous-titres complètement décalés sur Mecha Academy Zero ep 7 😭"]),
("2026-10-25 11:10","2026-10-25 12:30",2,["PH","ID"],["Google Play won't renew my Fanimation, BILL-207"])]
for a,b,lam_,cs_,txts in soc_inc:
    for w_ in range(iv(a)//3, iv(b)//3+1):
        for _ in range(rng.poisson(lam_)): P(TS15[w_],rng.choice(["Reddit","X"]),rng.choice(cs_),random.choice(txts))
soc=pd.DataFrame(posts,columns=["posted_at","platform","country_code","post_text"])
soc["posted_at"]=soc.posted_at+pd.to_timedelta(rng.integers(0,900,len(soc)),unit="s")
soc=soc.sort_values("posted_at").reset_index(drop=True)
soc["post_id"]=["SP-"+str(100000+i) for i in range(len(soc))]
soc["author_handle"]=["@"+random.choice(["otaku","animefan","weeb","senpai","kitsune","ronin","mecha"])+str(random.randint(10,9999)) for _ in range(len(soc))]
soc["engagement"]=rng.negative_binomial(2,0.05,len(soc))
soc=soc[["post_id","posted_at","platform","author_handle","country_code","post_text","engagement"]]

svc_dom={"auth-service":"Authentication","payments-router":"Billing & Payments","billing-api":"Billing & Payments","playback-service":"Playback & CDN","cdn-config":"Playback & CDN","player-sdk-androidtv":"Playback & CDN","content-pipeline":"Content Ingestion","subtitle-service":"Subtitles & Localization","app-ios":"Device Apps","app-android":"Device Apps","app-roku":"Device Apps","search-api":"Search & Discovery"}
ch=[]
for d in range(15):
    for _ in range(rng.integers(4,9)):
        s=random.choice(list(svc_dom)); t=start+pd.Timedelta(days=d,minutes=int(rng.integers(0,1440)))
        ch.append((t,s,svc_dom[s],"deployment",f"v{random.randint(1,4)}.{random.randint(0,20)}.{random.randint(0,9)}","Routine release","Global","success"))
ch+=[(pd.Timestamp("2026-10-27 08:55"),"auth-service","Authentication","config_change","cfg-8812","Token cache TTL change for APAC edge","APAC","rolled_back"),
(pd.Timestamp("2026-10-27 09:44"),"auth-service","Authentication","rollback","cfg-8811","Revert token cache TTL change","APAC","success"),
(pd.Timestamp("2026-10-27 15:30"),"player-sdk-androidtv","Playback & CDN","deployment","v2.3.0","Android TV player SDK: new connection pooling for LATAM CDN edges","LATAM","rolled_back"),
(pd.Timestamp("2026-10-27 17:18"),"player-sdk-androidtv","Playback & CDN","rollback","v2.2.9","Rollback Android TV player SDK to v2.2.9","LATAM","success"),
(pd.Timestamp("2026-10-27 17:55"),"content-pipeline","Content Ingestion","content_publish","job-55120","Publish Spirit Ronin S1E3 + subtitle packs (es-419 pack failed validation silently)","Global","partial"),
(pd.Timestamp("2026-10-27 20:02"),"subtitle-service","Subtitles & Localization","content_publish","job-55188","Republish es-419 subtitle pack for Spirit Ronin S1E3","LATAM","success"),
(pd.Timestamp("2026-10-27 19:55"),"payments-router","Billing & Payments","config_change","cfg-9031","Migrate Adyen EU traffic to new regional endpoint","EMEA","rolled_back"),
(pd.Timestamp("2026-10-27 22:05"),"payments-router","Billing & Payments","rollback","cfg-9030","Revert Adyen EU endpoint migration","EMEA","success"),
(pd.Timestamp("2026-10-16 01:52"),"auth-service","Authentication","deployment","v5.2.0","Session token signing key rotation","NA","rolled_back"),
(pd.Timestamp("2026-10-20 18:58"),"cdn-config","Playback & CDN","config_change","cfg-7710","EU-West CDN edge capacity rebalancing","EMEA","rolled_back"),
(pd.Timestamp("2026-10-23 14:50"),"content-pipeline","Content Ingestion","content_publish","job-54002","Publish Mecha Academy Zero E7 (fr-FR subtitle timing offset)","Global","partial"),
(pd.Timestamp("2026-10-25 10:55"),"billing-api","Billing & Payments","deployment","v3.8.1","Google Play billing library upgrade","APAC","rolled_back")]
chg=pd.DataFrame(ch,columns=["change_ts","service","domain","change_type","version","description","scope","outcome"]).sort_values("change_ts").reset_index(drop=True)
chg.insert(0,"change_id",["CHG-"+str(20000+i) for i in range(len(chg))])
incs=[("INC-1016-01","Login failures in North America (AUTH-401)","Authentication","SEV1","US,CA","NA","2026-10-16 02:22","2026-10-16 02:28","2026-10-16 03:35","Monitoring","Session token signing key rotation invalidated active sessions"),
("INC-1020-01","EU-West CDN edge degradation - buffering","Playback & CDN","SEV2","GB,FR","EMEA","2026-10-20 19:04","2026-10-20 19:08","2026-10-20 20:20","Monitoring","CDN edge capacity rebalancing overloaded two PoPs"),
("INC-1023-01","French subtitles out of sync - Mecha Academy Zero E7","Subtitles & Localization","SEV3","FR","EMEA",None,"2026-10-23 16:10","2026-10-23 17:50","Support escalation","fr-FR subtitle pack published with timing offset"),
("INC-1025-01","Google Play renewals failing in PH/ID (BILL-207)","Billing & Payments","SEV2","PH,ID","APAC","2026-10-25 11:35","2026-10-25 11:40","2026-10-25 12:35","Monitoring","Google Play billing library upgrade regression"),
("INC-1027-01","APAC login failures (AUTH-503)","Authentication","SEV1","JP,IN,AU,PH,ID,KR","APAC","2026-10-27 09:04","2026-10-27 09:06","2026-10-27 09:50","Monitoring","Token cache TTL config change at APAC edge"),
("INC-1027-02","Android TV playback failures in LATAM (Error 1002)","Playback & CDN","SEV1","BR,MX,AR","LATAM","2026-10-27 16:22","2026-10-27 16:25","2026-10-27 17:35","Monitoring (delayed)","Android TV player SDK v2.3.0 connection pooling silently dropping sessions"),
("INC-1027-03","Spanish subtitles missing - Spirit Ronin E3","Subtitles & Localization","SEV3","MX,ES,AR,CO,CL","LATAM,EMEA",None,"2026-10-27 19:40","2026-10-27 20:25","Support escalation","es-419 subtitle pack failed validation silently during publish"),
("INC-1027-04","Adyen payment failures in DE/AT","Billing & Payments","SEV2","DE,AT","EMEA","2026-10-27 20:45","2026-10-27 20:50","2026-10-27 22:15","Monitoring","Adyen EU endpoint migration misrouted authorizations")]
inc_df=pd.DataFrame(incs,columns=["incident_id","title","domain","severity","countries","region_groups","first_alert_at","declared_at","resolved_at","detected_by","root_cause"])
for c_ in ["first_alert_at","declared_at","resolved_at"]: inc_df[c_]=pd.to_datetime(inc_df[c_])

def save(pdf,name): spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{FQ}.{name}")
for pdf,name in [(cdf.drop(columns=["subscribers_k"]),"regions"),(rel_df,"release_schedule"),(cs_df,"concurrent_streams"),(sm,"service_metrics"),
                 (pay_df,"payment_transactions"),(tickets,"support_tickets"),(soc,"social_posts"),(chg,"change_events"),(inc_df,"incidents")]:
    save(pdf,name); print("wrote",name,len(pdf))